# Whisper-small Hindi Fine-tuning  🎙️

**Before running:** `Runtime → Change runtime type → T4 GPU`

| Step | What happens |
|------|-------------|
| 1 | Check GPU |
| 2 | Install packages |
| 3 | HuggingFace login |
| 4 | Upload `train_manifest.json` |
| 5 | Download audio & build dataset _(one-time, ~30–45 min)_ |
| 6 | Train Whisper-small _(~1–2 hours on T4)_ |
| 7 | Download fine-tuned model |
| 8 | Quick sanity-check inference |

In [ ]:
# ── Step 1: Verify GPU ────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────
!pip install -q \
    transformers datasets accelerate \
    soundfile librosa \
    jiwer tqdm \
    huggingface_hub
print('✓ Packages installed')

In [ ]:
# ── Step 3: HuggingFace login ────────────────────────────────
from huggingface_hub import login
login()   # paste your token from https://huggingface.co/settings/tokens

In [ ]:
# ── Step 4: Upload train_manifest.json ───────────────────────
# File is at:  q1_finetune/processed/train_manifest.json  (~3 MB)
from google.colab import files
import json

uploaded      = files.upload()
manifest_path = list(uploaded.keys())[0]

with open(manifest_path, encoding='utf-8') as f:
    manifest = json.load(f)

print(f'✓ Loaded {len(manifest):,} segments')

In [ ]:
# ── Step 5: Download audio & extract features ────────────────
# Runs once per Colab session; cached to /content/dataset_cache
# If the session restarts, skip to the cell below this one.

import io, requests, numpy as np, soundfile as sf
from tqdm.notebook import tqdm
from transformers import WhisperProcessor
from datasets import Dataset, DatasetDict

MODEL_ID  = 'openai/whisper-small'
LANGUAGE  = 'Hindi'
TASK      = 'transcribe'
CACHE_DIR = '/content/dataset_cache'

processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

# 90 / 10 train / val split (same seed as local train.py)
rec_ids   = list({m['recording_id'] for m in manifest})
np.random.seed(42)
np.random.shuffle(rec_ids)
train_ids = set(rec_ids[:int(len(rec_ids) * 0.9)])

train_entries = [e for e in manifest if     e['recording_id'] in train_ids]
val_entries   = [e for e in manifest if not e['recording_id'] in train_ids]
print(f'Train: {len(train_entries):,}  |  Val: {len(val_entries):,}')

def featurise(entries, split_name):
    rows, skipped = [], 0
    for entry in tqdm(entries, desc=split_name):
        try:
            resp = requests.get(entry['audio_url'], timeout=60)
            resp.raise_for_status()
            data, sr = sf.read(io.BytesIO(resp.content))
            seg = data[int(entry['start']*sr) : int(entry['end']*sr)]
            if seg.ndim > 1: seg = seg.mean(axis=1)
            if sr != 16000:
                import librosa
                seg = librosa.resample(seg.astype(np.float32), orig_sr=sr, target_sr=16000)
                sr  = 16000
            feats  = processor.feature_extractor(
                seg.astype(np.float32), sampling_rate=sr, return_tensors='pt'
            ).input_features[0]
            labels = processor.tokenizer(entry['text']).input_ids
            rows.append({'input_features': feats, 'labels': labels})
        except:
            skipped += 1
    print(f'{split_name}: {len(rows):,} ok  |  {skipped} skipped')
    return rows

train_ds = Dataset.from_list(featurise(train_entries, 'Train'))
val_ds   = Dataset.from_list(featurise(val_entries,   'Val  '))
dataset  = DatasetDict({'train': train_ds, 'validation': val_ds})

dataset.save_to_disk(CACHE_DIR)
print(f'✓ Dataset cached → {CACHE_DIR}')

In [ ]:
# ── Step 5b: (Re)load from cache if session restarted ────────
# Run THIS cell instead of Step 5 if you already downloaded the audio.

# from datasets import DatasetDict
# from transformers import WhisperProcessor
# MODEL_ID, LANGUAGE, TASK = 'openai/whisper-small', 'Hindi', 'transcribe'
# processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
# dataset   = DatasetDict.load_from_disk('/content/dataset_cache')
# print(f"Train: {len(dataset['train']):,}  |  Val: {len(dataset['validation']):,}")

In [ ]:
# ── Step 6: Train ────────────────────────────────────────────
import torch, transformers
from dataclasses import dataclass
from typing import Any, Dict, List
from jiwer import wer as jiwer_wer
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# ── Data collator ──────────────────────────────────────────────
@dataclass
class DataCollator:
    processor: Any
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{'input_features': f['input_features']} for f in features],
            return_tensors='pt')
        lb = self.processor.tokenizer.pad(
            [{'input_ids': f['labels']} for f in features],
            return_tensors='pt')
        labels = lb['input_ids'].masked_fill(lb.attention_mask.ne(1), -100)
        if (labels[:, 0] == processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

# ── WER metric ─────────────────────────────────────────────────
def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = [' '.join(s.split()) for s in
                 processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)]
    label_str = [' '.join(s.split()) for s in
                 processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {'wer': round(jiwer_wer(label_str, pred_str), 4)}

# ── Model ──────────────────────────────────────────────────────
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.config.suppress_tokens    = []
model.config.use_cache           = False

# ── Training args ──────────────────────────────────────────────
_new_api = tuple(int(x) for x in transformers.__version__.split('.')[:2]) >= (4, 41)
_kw = dict(
    output_dir                 = '/content/whisper-small-hindi-ft',
    per_device_train_batch_size= 16,
    gradient_accumulation_steps= 1,
    learning_rate              = 1e-5,
    warmup_steps               = 200,
    max_steps                  = 2000,
    gradient_checkpointing     = True,
    fp16                       = True,
    per_device_eval_batch_size = 8,
    predict_with_generate      = True,
    generation_max_length      = 225,
    save_steps                 = 500,
    eval_steps                 = 500,
    logging_steps              = 25,
    report_to                  = ['tensorboard'],
    load_best_model_at_end     = True,
    metric_for_best_model      = 'wer',
    greater_is_better          = False,
    push_to_hub                = False,
)
_kw['eval_strategy' if _new_api else 'evaluation_strategy'] = 'steps'

training_args = Seq2SeqTrainingArguments(**_kw)

# ── Trainer ────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    args            = training_args,
    model           = model,
    train_dataset   = dataset['train'],
    eval_dataset    = dataset['validation'],
    data_collator   = DataCollator(processor=processor),
    compute_metrics = compute_metrics,
    tokenizer       = processor.feature_extractor,
)

print('Starting training …')
trainer.train()

In [ ]:
# ── Step 7: Save & download fine-tuned model ─────────────────
from google.colab import files

trainer.save_model('/content/whisper-small-hindi-ft')
processor.save_pretrained('/content/whisper-small-hindi-ft')
print('✓ Model saved')

!zip -r /content/whisper-small-hindi-ft.zip /content/whisper-small-hindi-ft
files.download('/content/whisper-small-hindi-ft.zip')
print('✓ Download started')

In [ ]:
# ── Step 8: Sanity-check inference on 5 val samples ──────────
import torch
model.eval()
forced = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

for row in dataset['validation'].select(range(5)):
    feats = torch.tensor(row['input_features']).unsqueeze(0).to('cuda')
    with torch.no_grad():
        ids = model.generate(feats, forced_decoder_ids=forced)
    hyp = processor.tokenizer.decode(ids[0], skip_special_tokens=True)
    ref = processor.tokenizer.decode(row['labels'], skip_special_tokens=True)
    print(f'REF : {ref}')
    print(f'HYP : {hyp}')
    print()